In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# %skip
# dbutils.widgets.text('catalog', 'de_dev')

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.trend_analysis AS

WITH daily_sales AS (

SELECT

sale_date,

SUM(sale_amount) total_revenue,

SUM(quantity) total_quantity,

COUNT(*) total_orders

FROM ${catalog}.silver.sales_scd_1

GROUP BY sale_date

)

SELECT

sale_date,

total_revenue,

total_quantity,

total_orders,

-- Running Revenue

SUM(total_revenue)

OVER(

ORDER BY sale_date

)

AS running_revenue,

-- Running Quantity

SUM(total_quantity)

OVER(

ORDER BY sale_date

)

AS running_quantity,

-- Running Orders

SUM(total_orders)

OVER(

ORDER BY sale_date

)

AS running_orders,

-- 7 Day Moving Average

AVG(total_revenue)

OVER(

ORDER BY sale_date

ROWS BETWEEN 6 PRECEDING

AND CURRENT ROW

)

AS moving_avg_7_days,

-- 30 Day Moving Average

AVG(total_revenue)

OVER(

ORDER BY sale_date

ROWS BETWEEN 29 PRECEDING

AND CURRENT ROW

)

AS moving_avg_30_days,

-- Cumulative Revenue

SUM(total_revenue)

OVER(

ORDER BY sale_date

ROWS BETWEEN UNBOUNDED PRECEDING

AND CURRENT ROW

)

AS cumulative_revenue

FROM daily_sales

ORDER BY sale_date;